In [44]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
import validation
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import make_pipeline



In [45]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
import validation
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import make_pipeline



In [46]:
old_to_predict_mens = pd.read_csv("old_data_to_predict_mens.csv")

In [47]:
old_to_predict_mens.sort_values(by="t1_adj_margin", ascending=False)[["Season", "t1_TeamName", "t1_adj_margin"]]

,Season,t1_TeamName,t1_adj_margin
1100,2019,Gonzaga,42.376293
1056,2019,Gonzaga,42.376293
1086,2019,Gonzaga,42.376293
2423,2019,Gonzaga,42.376293
833,2015,Kentucky,41.987377
...,...,...,...
1646,2008,MS Valley St,-11.968847
2366,2019,NC Central,-14.442869
1344,2003,UNC Asheville,-14.910886
0,2003,UNC Asheville,-14.910886


In [48]:
to_predict_mens = pd.read_csv("to_predict_mens.csv")

In [50]:
to_predict_mens.sort_values(by="t1_adj_margin", ascending=False)[["t1_TeamName", "Season"]]

,t1_TeamName,Season
1082,Gonzaga,2019
1052,Gonzaga,2019
1096,Gonzaga,2019
2474,Gonzaga,2019
831,Kentucky,2015
...,...,...
1699,MS Valley St,2008
2417,NC Central,2019
1399,UNC Asheville,2003
0,UNC Asheville,2003


### Evaluate Impact on First Round Stats Model

In [51]:
# Define the classifier and parameter grid
model = LogisticRegression(C=0.05)
pipeline = make_pipeline(StandardScaler(), model)
param_grid = {
    'logisticregression__C': [.005, 0.001, .05, 0.01, 0.1],
}

In [52]:
to_predict_mens_first_round = to_predict_mens[(to_predict_mens["GameRound"] == 1)
                                                    & (to_predict_mens.final_odds.notnull())]

In [53]:
to_predict_mens_first_round_recent = to_predict_mens_first_round[to_predict_mens_first_round.Season >= 2009]

In [54]:
baseline_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank']
eval_df = validation.run_evaluation_framework(to_predict_mens_first_round_recent, pipeline, baseline_features, param_grid, cv_start=2013)

[[ 0.2337584  -0.2337584   0.34084912 -0.34084912 -0.23104064  0.23104064]]
[[ 0.21845653 -0.21845653  0.35346032 -0.35346032 -0.21916828  0.21916828]]
[[ 0.26840578 -0.26840578  0.36974363 -0.36974363 -0.17939435  0.17939435]]
[[ 0.31123128 -0.31123128  0.37873249 -0.37873249 -0.16810367  0.16810367]]
[[ 0.33365018 -0.33365018  0.32877817 -0.32877817 -0.20264398  0.20264398]]
[[ 0.37192779 -0.37192779  0.32949371 -0.32949371 -0.24065543  0.24065543]]
[[ 0.36631885 -0.36631885  0.3409908  -0.3409908  -0.21386316  0.21386316]]
[[ 0.33186188 -0.33186188  0.38187788 -0.38187788 -0.20906324  0.20906324]]
[[ 0.33246849 -0.33246849  0.36068422 -0.36068422 -0.21702876  0.21702876]]
[[ 0.33090926 -0.33090926  0.33840534 -0.33840534 -0.25303719  0.25303719]]
[[ 0.33964378 -0.33964378  0.29647986 -0.29647986 -0.29398078  0.29398078]]


In [55]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.172153,"(-0.1743702424456949, -0.16993507359843749)",0.169975


In [94]:
baseline_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank']
eval_df = validation.run_evaluation_framework(to_predict_mens_first_round, pipeline, baseline_features, param_grid, cv_start=2013)

[[ 0.36082747 -0.36082747  0.50030652 -0.50030652 -0.10925892  0.10925892]]
[[ 0.33889971 -0.33889971  0.49700295 -0.49700295 -0.10732782  0.10732782]]
[[ 0.36898262 -0.36898262  0.49428871 -0.49428871 -0.08444329  0.08444329]]
[[ 0.39376659 -0.39376659  0.49154468 -0.49154468 -0.07865738  0.07865738]]
[[ 0.41347118 -0.41347118  0.43771694 -0.43771694 -0.10482407  0.10482407]]
[[ 0.43234894 -0.43234894  0.43952885 -0.43952885 -0.13130526  0.13130526]]
[[ 0.42451363 -0.42451363  0.43983262 -0.43983262 -0.11877523  0.11877523]]
[[ 0.39083186 -0.39083186  0.47146589 -0.47146589 -0.11913866  0.11913866]]
[[ 0.39134702 -0.39134702  0.44717619 -0.44717619 -0.12820832  0.12820832]]
[[ 0.3954227  -0.3954227   0.41799073 -0.41799073 -0.15747399  0.15747399]]
[[ 0.39935631 -0.39935631  0.38062515 -0.38062515 -0.19135077  0.19135077]]


In [95]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.16788,"(-0.1698428911787427, -0.1659161671099005)",0.171663


In [56]:
new_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank',
                't1_injured_players_value', 't2_injured_players_value']

eval_df = validation.run_evaluation_framework(to_predict_mens_first_round_recent, pipeline, new_features, param_grid, cv_start=2013)

[[ 0.23376956 -0.23376956  0.34106127 -0.34106127 -0.23137721  0.23137721
  -0.00411621  0.00411621]]
[[ 0.21649059 -0.21649059  0.35539873 -0.35539873 -0.22849706  0.22849706
  -0.08426089  0.08426089]]
[[ 0.26293659 -0.26293659  0.37224541 -0.37224541 -0.19848543  0.19848543
  -0.16309771  0.16309771]]
[[ 0.30764951 -0.30764951  0.37931195 -0.37931195 -0.18025697  0.18025697
  -0.12106042  0.12106042]]
[[ 0.331052   -0.331052    0.33330802 -0.33330802 -0.217068    0.217068
  -0.15892066  0.15892066]]
[[ 0.36932042 -0.36932042  0.33418666 -0.33418666 -0.25757899  0.25757899
  -0.17283896  0.17283896]]
[[ 0.35500702 -0.35500702  0.3521425  -0.3521425  -0.23547735  0.23547735
  -0.18689515  0.18689515]]
[[ 0.31758918 -0.31758918  0.39345717 -0.39345717 -0.23577197  0.23577197
  -0.18880717  0.18880717]]
[[ 0.31403865 -0.31403865  0.37553527 -0.37553527 -0.24572774  0.24572774
  -0.17672478  0.17672478]]
[[ 0.31632728 -0.31632728  0.35038831 -0.35038831 -0.28230215  0.28230215
  -0.16803

In [57]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.171112,"(-0.17351456499874046, -0.16870844866746298)",0.168546


In [64]:
new_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank',
                't1_health_score', 't2_health_score']

In [65]:
eval_df = validation.run_evaluation_framework(to_predict_mens_first_round_recent, pipeline, baseline_features, param_grid, cv_start=2013)

[[ 0.2337584  -0.2337584   0.34084912 -0.34084912 -0.23104064  0.23104064]]
[[ 0.21845653 -0.21845653  0.35346032 -0.35346032 -0.21916828  0.21916828]]
[[ 0.26840578 -0.26840578  0.36974363 -0.36974363 -0.17939435  0.17939435]]
[[ 0.31123128 -0.31123128  0.37873249 -0.37873249 -0.16810367  0.16810367]]
[[ 0.33365018 -0.33365018  0.32877817 -0.32877817 -0.20264398  0.20264398]]
[[ 0.37192779 -0.37192779  0.32949371 -0.32949371 -0.24065543  0.24065543]]
[[ 0.36631885 -0.36631885  0.3409908  -0.3409908  -0.21386316  0.21386316]]
[[ 0.33186188 -0.33186188  0.38187788 -0.38187788 -0.20906324  0.20906324]]
[[ 0.33246849 -0.33246849  0.36068422 -0.36068422 -0.21702876  0.21702876]]
[[ 0.33090926 -0.33090926  0.33840534 -0.33840534 -0.25303719  0.25303719]]
[[ 0.33964378 -0.33964378  0.29647986 -0.29647986 -0.29398078  0.29398078]]


In [66]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.172235,"(-0.17455815435703176, -0.16991280953173538)",0.169975


In [67]:
new_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank',
                't1_health_score', 't2_health_score', 't1_injured_players_value', 't2_injured_players_value']

In [68]:
eval_df = validation.run_evaluation_framework(to_predict_mens_first_round_recent, pipeline, baseline_features, param_grid, cv_start=2013)

[[ 0.2337584  -0.2337584   0.34084912 -0.34084912 -0.23104064  0.23104064]]
[[ 0.21845653 -0.21845653  0.35346032 -0.35346032 -0.21916828  0.21916828]]
[[ 0.26840578 -0.26840578  0.36974363 -0.36974363 -0.17939435  0.17939435]]
[[ 0.31123128 -0.31123128  0.37873249 -0.37873249 -0.16810367  0.16810367]]
[[ 0.33365018 -0.33365018  0.32877817 -0.32877817 -0.20264398  0.20264398]]
[[ 0.37192779 -0.37192779  0.32949371 -0.32949371 -0.24065543  0.24065543]]
[[ 0.36631885 -0.36631885  0.3409908  -0.3409908  -0.21386316  0.21386316]]
[[ 0.33186188 -0.33186188  0.38187788 -0.38187788 -0.20906324  0.20906324]]
[[ 0.33246849 -0.33246849  0.36068422 -0.36068422 -0.21702876  0.21702876]]
[[ 0.33090926 -0.33090926  0.33840534 -0.33840534 -0.25303719  0.25303719]]
[[ 0.33964378 -0.33964378  0.29647986 -0.29647986 -0.29398078  0.29398078]]


In [69]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.172104,"(-0.17439802827074705, -0.16981009323158883)",0.169975


In [75]:
odds_features = ['final_odds']


first_round_df = to_predict_mens[(to_predict_mens.GameRound == 1)].copy()


first_round_df = first_round_df[~first_round_df.final_odds.isna()]

In [92]:
eval_df = validation.run_evaluation_framework(first_round_df, pipeline, odds_features, param_grid, cv_start=2013)

[[-1.59826703]]
[[-1.5784245]]
[[-1.60523263]]
[[-1.64188448]]
[[-1.63033733]]
[[-1.70001017]]
[[-1.678187]]
[[-1.68835753]]
[[-1.66308601]]
[[-1.64665667]]
[[-1.62507845]]


In [93]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.1},-0.16766,"(-0.1694484811181598, -0.16587151531125582)",0.172134


The new injury feature improves the statistics model.

Odds is worse though, which doesn't seem right 

Best New Injury Features: -0.171112, 0.168546

Baseline Odds Model (using all data, but eval rolling season on same years): -0.167682, 0.172134

Baseline Stats Model: -0.172153, 0.169975

Baseline Stat Model using all data _, 0.171663

Caveat - of course there's some leakage here since we identified missing players by whether or not they played, which we wouldn't know for sure until the games


### Evaluate Impact on Overall Model

In [96]:
to_predict_mens_recent = to_predict_mens[(to_predict_mens.Season >= 2009)
        # filter out first four games
        & (to_predict_mens.t1_injured_players_value.notnull())
        & (to_predict_mens.t2_injured_players_value.notnull())
        ]

In [97]:
baseline_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank']
eval_df = validation.run_evaluation_framework(to_predict_mens_recent, pipeline, baseline_features, param_grid, cv_start=2013)

[[ 0.18504594 -0.18504594  0.41631808 -0.41631808 -0.27547432  0.27547432]]
[[ 0.15945357 -0.15945357  0.42621724 -0.42621724 -0.30133648  0.30133648]]
[[ 0.21685382 -0.21685382  0.38983144 -0.38983144 -0.26675001  0.26675001]]
[[ 0.27383244 -0.27383244  0.3937382  -0.3937382  -0.26366595  0.26366595]]
[[ 0.28519005 -0.28519005  0.38747553 -0.38747553 -0.2665695   0.2665695 ]]
[[ 0.302364   -0.302364    0.39021542 -0.39021542 -0.28428024  0.28428024]]
[[ 0.2814681  -0.2814681   0.40850747 -0.40850747 -0.24806287  0.24806287]]
[[ 0.254847   -0.254847    0.47028895 -0.47028895 -0.22920867  0.22920867]]
[[ 0.24443685 -0.24443685  0.45992629 -0.45992629 -0.23703734  0.23703734]]
[[ 0.23448816 -0.23448816  0.42899185 -0.42899185 -0.2677185   0.2677185 ]]
[[ 0.26759639 -0.26759639  0.39718016 -0.39718016 -0.2487073   0.2487073 ]]


In [98]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.1865,"(-0.18813079557474038, -0.1848694772431509)",0.186888


In [99]:
to_predict_mens_recent[to_predict_mens_recent.t1_injured_players_value.isna()][["Season", "t1_TeamName"]]

,Season,t1_TeamName


In [100]:
to_predict_mens_recent.shape

(1880, 59)

In [101]:
new_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank',
                't1_injured_players_value', 't2_injured_players_value']

eval_df = validation.run_evaluation_framework(to_predict_mens_recent, pipeline, new_features, param_grid, cv_start=2013)

[[ 0.1828972  -0.1828972   0.42014671 -0.42014671 -0.28073513  0.28073513
  -0.03417147  0.03417147]]
[[ 0.15479665 -0.15479665  0.43336545 -0.43336545 -0.31379982  0.31379982
  -0.07853545  0.07853545]]
[[ 0.20690107 -0.20690107  0.40402904 -0.40402904 -0.28847901  0.28847901
  -0.13224635  0.13224635]]
[[ 0.26724356 -0.26724356  0.40055639 -0.40055639 -0.27896019  0.27896019
  -0.09885833  0.09885833]]
[[ 0.27899657 -0.27899657  0.39726955 -0.39726955 -0.27696625  0.27696625
  -0.10915333  0.10915333]]
[[ 0.29406135 -0.29406135  0.3975549  -0.3975549  -0.29668446  0.29668446
  -0.0871989   0.0871989 ]]
[[ 0.27254377 -0.27254377  0.41609318 -0.41609318 -0.25431122  0.25431122
  -0.05985132  0.05985132]]
[[ 0.24332758 -0.24332758  0.47909594 -0.47909594 -0.23839004  0.23839004
  -0.06501433  0.06501433]]
[[ 0.23102186 -0.23102186  0.47164954 -0.47164954 -0.24702681  0.24702681
  -0.07152111  0.07152111]]
[[ 0.22134509 -0.22134509  0.44123838 -0.44123838 -0.27713066  0.27713066
  -0.073

In [102]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.186142,"(-0.18770587104089648, -0.1845780448838703)",0.186463


In [ ]:
new_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank',
                't1_health_score', 't2_health_score']

eval_df = validation.run_evaluation_framework(to_predict_mens_recent, pipeline, new_features, param_grid, cv_start=2013)

[[ 0.18475748 -0.18475748  0.41678778 -0.41678778 -0.27573783  0.27573783
   0.00693694 -0.00693694]]
[[ 0.1569149  -0.1569149   0.42975453 -0.42975453 -0.305738    0.305738
   0.07451818 -0.07451818]]
[[ 0.21418862 -0.21418862  0.3972358  -0.3972358  -0.27097168  0.27097168
   0.11166623 -0.11166623]]
[[ 0.27291673 -0.27291673  0.3969456  -0.3969456  -0.26573036  0.26573036
   0.06417177 -0.06417177]]
[[ 0.28506229 -0.28506229  0.39066174 -0.39066174 -0.26693474  0.26693474
   0.06566814 -0.06566814]]
[[ 0.30312419 -0.30312419  0.39347262 -0.39347262 -0.28630139  0.28630139
   0.08032274 -0.08032274]]
[[ 0.2782072  -0.2782072   0.41637427 -0.41637427 -0.24946925  0.24946925
   0.08479285 -0.08479285]]
[[ 0.24732518 -0.24732518  0.48049046 -0.48049046 -0.23226139  0.23226139
   0.08329435 -0.08329435]]
[[ 0.23687299 -0.23687299  0.4691337  -0.4691337  -0.24086597  0.24086597
   0.08105443 -0.08105443]]
[[ 0.22669156 -0.22669156  0.43983396 -0.43983396 -0.2705012   0.2705012
   0.081491

In [ ]:
# eff margin: 0.190491
# final rank: 0.189166
# ordinal rank 0.206132
# injury 0.249987
# combined: 0.186615


In [104]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.18634,"(-0.18787629970332526, -0.18480300375985775)",0.186615


In [105]:
# Compare with rolling for using more data

eval_df = validation.run_evaluation_framework(to_predict_mens[to_predict_mens.GameRound != 0], pipeline, baseline_features, param_grid, cv_start=2013)


[[ 0.32879604 -0.32879604  0.35598369 -0.35598369 -0.1502124   0.1502124 ]]
[[ 0.32033849 -0.32033849  0.3568756  -0.3568756  -0.16329544  0.16329544]]
[[ 0.33291835 -0.33291835  0.34754006 -0.34754006 -0.15812473  0.15812473]]
[[ 0.34996439 -0.34996439  0.35661427 -0.35661427 -0.16172153  0.16172153]]
[[ 0.3526476  -0.3526476   0.35584754 -0.35584754 -0.16805426  0.16805426]]
[[ 0.35822102 -0.35822102  0.3597985  -0.3597985  -0.18068692  0.18068692]]
[[ 0.35189746 -0.35189746  0.36288078 -0.36288078 -0.16768729  0.16768729]]
[[ 0.3474748  -0.3474748   0.38538986 -0.38538986 -0.16212538  0.16212538]]
[[ 0.34072006 -0.34072006  0.38146796 -0.38146796 -0.16375685  0.16375685]]
[[ 0.33356556 -0.33356556  0.3664942  -0.3664942  -0.18143313  0.18143313]]
[[ 0.34157552 -0.34157552  0.35634944 -0.35634944 -0.17497844  0.17497844]]


In [106]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.01},-0.183211,"(-0.1844771552897655, -0.18194538388478804)",0.186995


Not as big of an impact of downstream rounds, but still does make a difference 

Baseline Stats Model:-0.1865, 0.186888

Baseline Stats using all data _, 0.186995

Best New Injury Features: -0.186142, 0.186463

Interesting, there's some evidence here that using only more recent data might be better

---

Overall, I care most about the rolling season cv, and the injury features seem to improve the model on all apples to apples comparions 
